# __2- Model Training__ 

Install torch library

 <span style="color: red;"> RUN IT ONCE IF YOU FACED ERROR IN CELL 3</span>

In [4]:
!pip install torch torchvision

  Using cached torch-2.10.0-cp311-cp311-win_amd64.whl.metadata (31 kB)
  Using cached torchvision-0.25.0-cp311-cp311-win_amd64.whl.metadata (5.4 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
Using cached torch-2.10.0-cp311-cp311-win_amd64.whl (113.7 MB)
Using cached torchvision-0.25.0-cp311-cp311-win_amd64.whl (4.0 MB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\rahaf\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
import torch
print(torch.__version__)


2.10.0+cpu


## 2.1 Normlization

In [6]:
import torch # pytorch main libirary
from torchvision import transforms # tp process images
from torch.utils.data import Dataset, DataLoader # Dataset to build a class for data , Dataloader to send data to the model as batches
from PIL import Image # to open images
import pandas as pd # to use train_df/val_df/test_df

transform = transforms.Compose([ # to make learning easier
    transforms.ToTensor(), # convert images from being in 0 to 255 to being 0 and 1
    transforms.Normalize( # this is ImageNet normlization
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

 Normliazation Equation = $$ \frac{x - mean}{std} = x_{normalized} $$

## 2.2 Dataset Class

 This class is needed to specify how to read images and how to return the image and its label.

In [7]:
class GestureDataset(Dataset):
    def __init__(self, dataframe, transform=None): # to store data & transformers
        self.dataframe = dataframe
        self.transform = transform

        self.label_map = { # Palm -> 0 , Fist -> 1
            "Palm": 0,
            "Fist": 1
        }

    def __len__(self): # Dataloader will use it to know the size of data
        return len(self.dataframe)

    def __getitem__(self, idx):                           # most importent function works like follows:
    
        img_path = self.dataframe.iloc[idx]["image_path"] # 1- take image path
        label = self.dataframe.iloc[idx]["label"]         # 2- open image

        image = Image.open(img_path).convert("RGB")       # 3- convert it to RGB

        if self.transform:                                # 4- apply transform
            image = self.transform(image)

        label = self.label_map[label]                     # 5- convert label to image

        return image, label                               # 6- return (image,label)


## 2.3 Creat Dataset for Each Split 

In [8]:
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("val.csv")
test_df = pd.read_csv("test.csv")

train_dataset = GestureDataset(train_df, transform=transform)
val_dataset = GestureDataset(val_df, transform=transform)
test_dataset = GestureDataset(test_df, transform=transform)

## 2.3 Creat DataLoder

In [9]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
import torch.nn as nn
from torchvision import models

# تحميل المودل
model = models.resnet18(pretrained=True)

# تعديل الطبقة الأخيرة
num_classes = 2
model.fc = nn.Linear(model.fc.in_features, num_classes)

# Loss
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)